# PlanetScope damage classification: RSP ResNet-50 Siamese comparison

This notebook is the MillionAID-pretrained RSP ResNet-50 counterpart to `2_training_evaluation.ipynb`. It consumes the same notebook-1 Planet NPZ/parquet pair and uses the shared preprocessing implementation: filtered footprint IDs, full-reference latitude bands, training-only shared pre/post percentile normalization and pair-consistent geometric augmentation.

RSP ResNet-50 does not need a stacking band, so the scratch experiment's stack and validation bands are merged:

| role | latitude-quantile band | use |
|---|---:|---|
| train | 0.55-1.00 | model weights |
| validation | 0.33-0.55 | early stopping, phase selection, threshold |
| test | 0.00-0.33 | one-time evaluation after selection |

Checkpoints and training phases are selected by validation ROC AUC. F1 is used only to fit the operating threshold after phase selection.

The scratch notebook's `0.33-0.44` validation band is additionally reported as `validation_core`, so validation diagnostics remain directly comparable. Final test footprints are identical.

The RSP checkpoint is RGB-pretrained. A scratch run using `bands=['R', 'G', 'B']` provides the strict pretraining comparison. The default four-band scratch run also uses NIR and therefore compares complete pipelines rather than initialization alone.


## 1. Colab setup and experiment configuration


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pyarrow geopandas rasterio scikit-learn


In [ ]:
import os
import sys
import json
import time
import copy
import hashlib
import importlib
import types
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader
from sklearn.metrics import (
    roc_auc_score, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay,
)

PROJECT_ROOT = '/content/drive/MyDrive/War-Damage-Detection'
PROJECT_DIR = os.path.join(PROJECT_ROOT, 'planet')
os.environ['PLANET_DAMAGE_BASE'] = PROJECT_DIR
os.environ['PLANET_EXPERIMENTS_DIR'] = os.path.join(PROJECT_DIR, 'experiments')
for path in (str(PROJECT_ROOT), str(PROJECT_DIR)):
    if path not in sys.path:
        sys.path.insert(0, path)

from planet import pipeline
importlib.reload(pipeline)
from planet.pipeline import (
    PlanetPairDataset, inspect_dataset, load_dataset,
    footprint_latitude_reference, latitude_quantile_spatial_split,
    spatial_split_metadata, estimate_shared_percentiles,
    best_f1_threshold, binary_metrics, experiment_dirs,
    atomic_json_dump, atomic_torch_save,
)

CONFIG = {
    'experiment_name': 'planet_exp_pretrained_rsp_resnet50_rgb_roc_auc',
    'seed': 0,
    'bands': ['R', 'G', 'B'],
    'resize_to': 128,
    'batch_size': 64,
    'num_workers': 2,
    # Expected location of the downloaded RSP-ResNet-50-E300 checkpoint.
    'rsp_checkpoint': os.path.join(
        PROJECT_DIR, 'data', 'pretrained', 'rsp-resnet-50-ckpt.pth'),
    'split': {
        'train': [0.55, 1.00],
        'stack': [0.44, 0.55],
        'val': [0.33, 0.44],
        'test': [0.00, 0.33],
    },
    'normalization': {
        'lower_percentile': 2.0,
        'upper_percentile': 98.0,
        'sample_patches': 20_000,
    },
    'augmentation': {
        'geometric': True,
        'brightness_jitter': 0.0,
    },
    'phase1': {
        'epochs': 20, 'learning_rate': 1e-3,
        'patience': 8, 'reduce_lr_patience': 4,
    },
    'phase2': {
        'epochs': 30, 'learning_rate': 1e-5,
        'trainable_backbone': 'top',
        'patience': 8, 'reduce_lr_patience': 4,
    },
    'selection_metric': 'roc_auc',
    'threshold_metric': 'F1',
}

pipeline.set_random_seed(CONFIG['seed'])
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
OUT = experiment_dirs(CONFIG['experiment_name'])
CONFIG_PATH = OUT['root'] / 'config.json'
FINAL_SELECTION_PATH = OUT['metrics'] / 'final_selection.json'
FINAL_MARKER_PATH = OUT['metrics'] / 'final_test_complete.json'
SEALED_EXPERIMENT = FINAL_MARKER_PATH.exists()
if CONFIG_PATH.exists():
    with CONFIG_PATH.open(encoding='utf-8') as handle:
        previous_config = json.load(handle)
    if previous_config != CONFIG:
        raise RuntimeError(
            'This experiment folder contains a different configuration. '
            'Choose a new experiment_name rather than overwriting it.')
else:
    atomic_json_dump(CONFIG, CONFIG_PATH)

print('torch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('device:', DEVICE)
print('experiment:', OUT['root'])
print('mode:', 'sealed evaluation' if SEALED_EXPERIMENT else 'train / resume')


## 2. Load the Planet dataset and reproduce notebook 2 geography


In [ ]:
dataset_info = inspect_dataset(validate_alignment=True)
data = load_dataset(load_images=True)
X = data['X']
labels = data['y'].astype(np.int8)
table = data['gdf'].reset_index(drop=True)

if len(X) != len(labels) or len(labels) != len(table):
    raise RuntimeError('NPZ/parquet row alignment changed after validation')
if not np.array_equal(
        data['system_index'], table['system:index'].astype(str).to_numpy()):
    raise RuntimeError('NPZ and parquet footprint IDs are not row-aligned')

reference = footprint_latitude_reference()
scratch_bands = {
    role: tuple(values) for role, values in CONFIG['split'].items()
}
scratch_split = latitude_quantile_spatial_split(
    table, bands=scratch_bands, reference=reference)

idx_train = scratch_split['train']
idx_val_core = scratch_split['val']
idx_val_extra = scratch_split['stack']
idx_val = np.sort(np.concatenate([idx_val_core, idx_val_extra]))
idx_test = scratch_split['test']

role = np.full(len(labels), 'unused', dtype='U10')
role[idx_train] = 'train'
role[idx_val] = 'val'
role[idx_test] = 'test'
if (role == 'unused').any():
    raise RuntimeError('Merged pretrained split left Planet rows unassigned')
if set(idx_train) & set(idx_val) or set(idx_train) & set(idx_test) or set(idx_val) & set(idx_test):
    raise RuntimeError('Spatial roles overlap')

scratch_meta = spatial_split_metadata(scratch_split)
split_meta = {
    'type': 'latitude_bands_pretrained_no_stack',
    'train': CONFIG['split']['train'],
    'validation': [0.33, 0.55],
    'validation_core': CONFIG['split']['val'],
    'validation_extra_from_stack': CONFIG['split']['stack'],
    'test': CONFIG['split']['test'],
    'buffer_m': 0.0,
    'reference_rows': scratch_meta['reference_rows'],
    'latitude_thresholds': scratch_meta['latitude_thresholds'],
    'counts': {
        'train': int(len(idx_train)),
        'validation': int(len(idx_val)),
        'validation_core': int(len(idx_val_core)),
        'validation_extra': int(len(idx_val_extra)),
        'test': int(len(idx_test)),
    },
}
atomic_json_dump(split_meta, OUT['metrics'] / 'spatial_split.json')

split_rows = []
for name, idx in (
    ('train', idx_train), ('validation', idx_val),
    ('validation_core', idx_val_core),
    ('validation_extra', idx_val_extra), ('test', idx_test),
):
    split_rows.append({
        'role': name, 'n': len(idx), 'fraction': len(idx) / len(labels),
        'damaged_fraction': (
            float(labels[idx].mean()) if name != 'test' else np.nan),
    })
display(pd.DataFrame(split_rows))


In [ ]:
colors = {'train': 'tab:blue', 'val': 'tab:orange', 'test': 'tab:red'}
fig, ax = plt.subplots(figsize=(6, 9))
for name, color in colors.items():
    mask = role == name
    ax.scatter(table.loc[mask, 'lon'], table.loc[mask, 'lat'],
               s=2, alpha=.35, color=color,
               label=f'{name} ({mask.sum():,})')
ax.set(
    xlabel='longitude', ylabel='latitude',
    title='RSP split: same train/test, merged validation',
)
ax.legend(markerscale=4)
fig.savefig(OUT['figures'] / 'spatial_split.png', dpi=180, bbox_inches='tight')
plt.show()


## 3. Shared Planet normalization and lazy paired loaders


In [ ]:
ncfg = CONFIG['normalization']
lo, hi, normalization_rows = estimate_shared_percentiles(
    X, idx_train, bands=CONFIG['bands'],
    lower=ncfg['lower_percentile'], upper=ncfg['upper_percentile'],
    sample_patches=ncfg['sample_patches'], seed=CONFIG['seed'])
normalization = {
    'type': 'shared_pre_post_training_percentiles',
    'bands': CONFIG['bands'],
    'lo': lo.tolist(), 'hi': hi.tolist(),
    'sampled_training_rows': int(len(normalization_rows)),
}
atomic_json_dump(normalization, OUT['metrics'] / 'normalization.json')
display(pd.DataFrame({'band': CONFIG['bands'], 'p02': lo, 'p98': hi}))


def make_loader(indices, training=False, batch_size=None):
    dataset = PlanetPairDataset(
        X, labels, indices, bands=CONFIG['bands'], lo=lo, hi=hi,
        augment=(training and CONFIG['augmentation']['geometric']),
        brightness_jitter=(
            CONFIG['augmentation']['brightness_jitter'] if training else 0.0),
    )
    generator = torch.Generator().manual_seed(CONFIG['seed'])
    return DataLoader(
        dataset,
        batch_size=int(batch_size or CONFIG['batch_size']),
        shuffle=training,
        num_workers=CONFIG['num_workers'],
        pin_memory=USE_AMP,
        persistent_workers=False,
        generator=generator,
    )


train_loader = make_loader(idx_train, training=True)
val_loader = make_loader(idx_val)
val_core_loader = make_loader(idx_val_core)
val_extra_loader = make_loader(idx_val_extra)

positives = int(labels[idx_train].sum())
negatives = len(idx_train) - positives
pos_weight = torch.tensor(
    negatives / max(positives, 1), dtype=torch.float32, device=DEVICE)
print(f'train intact={negatives:,}, damaged={positives:,}, '
      f'pos_weight={pos_weight.item():.3f}')


In [ ]:
sample_x, sample_y = next(iter(train_loader))
half = sample_x.shape[1] // 2
fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for column in range(6):
    axes[0, column].imshow(sample_x[column, :half].permute(1, 2, 0))
    axes[1, column].imshow(sample_x[column, half:].permute(1, 2, 0))
    axes[0, column].set_title(f'y={int(sample_y[column])}')
    axes[0, column].axis('off'); axes[1, column].axis('off')
axes[0, 0].set_ylabel('pre'); axes[1, 0].set_ylabel('post')
fig.suptitle('Augmented normalized RGB pairs seen by RSP ResNet-50')
plt.tight_layout(); plt.show()


## 4. Load the MillionAID-pretrained RSP ResNet-50 backbone


In [ ]:
def load_original_rsp_checkpoint(checkpoint_path):
    """Load the trusted original RSP file without its yacs dependency."""
    class Stub(dict):
        def __init__(self, *args, **kwargs):
            super().__init__(*(args[:1] or ({},)))

        def __getattr__(self, key):
            try:
                return self[key]
            except KeyError:
                raise AttributeError(key)

    stubbed = []
    for name in ('yacs', 'yacs.config'):
        if name not in sys.modules:
            module = types.ModuleType(name)
            module.CfgNode = Stub
            module.__getattr__ = lambda _name, cls=Stub: cls
            sys.modules[name] = module
            stubbed.append(name)
    try:
        return torch.load(
            checkpoint_path, map_location='cpu', weights_only=False)
    finally:
        for name in stubbed:
            del sys.modules[name]


rsp_source = Path(CONFIG['rsp_checkpoint'])
rsp_safe = (Path(PROJECT_DIR) / 'data' / 'pretrained' /
            'rsp-resnet-50-weights.pth')
rsp_safe.parent.mkdir(parents=True, exist_ok=True)
if not rsp_safe.exists():
    if not rsp_source.exists():
        raise FileNotFoundError(
            f'Missing RSP checkpoint: {rsp_source}. Edit CONFIG["rsp_checkpoint"] '
            'to point at rsp-resnet-50-ckpt.pth.')
    original = load_original_rsp_checkpoint(rsp_source)
    original_state = (
        original['model'] if isinstance(original, dict) and 'model' in original
        else original)
    torch.save({
        'model': dict(original_state),
        'epoch': int(original.get('epoch', -1)) if isinstance(original, dict) else -1,
        'max_accuracy': (
            float(original.get('max_accuracy', float('nan')))
            if isinstance(original, dict) else float('nan')),
    }, rsp_safe)
    del original
    print('wrote dependency-free RSP weights:', rsp_safe)

rsp_checkpoint = torch.load(rsp_safe, map_location='cpu', weights_only=True)
rsp_state = rsp_checkpoint['model']
rsp_state = {
    (name[7:] if name.startswith('module.') else name): value
    for name, value in rsp_state.items()
    if not (name[7:] if name.startswith('module.') else name).startswith('fc.')
}
digest = hashlib.sha256()
with rsp_safe.open('rb') as handle:
    for chunk in iter(lambda: handle.read(1024 * 1024), b''):
        digest.update(chunk)
rsp_sha256 = digest.hexdigest()
print(
    f'RSP weights: {len(rsp_state)} tensors, '
    f'epoch={rsp_checkpoint.get("epoch", "?")}, sha256={rsp_sha256[:12]}…')


def new_rsp_backbone():
    backbone = torchvision.models.resnet50(weights=None)
    backbone.fc = nn.Identity()
    missing, unexpected = backbone.load_state_dict(rsp_state, strict=False)
    if missing or unexpected:
        raise RuntimeError(
            f'RSP/torchvision architecture mismatch; missing={missing[:5]}, '
            f'unexpected={unexpected[:5]}')
    return backbone


## 5. Siamese RSP model and resumable two-phase training


In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
FEATURE_DIM = 2048


class RspSiameseDamage(nn.Module):
    def __init__(self, backbone, resize_to, bands):
        super().__init__()
        if list(bands) != ['R', 'G', 'B']:
            raise ValueError('RSP ResNet-50 expects bands in RGB order')
        self.backbone = backbone
        self.resize_to = int(resize_to)
        self.half_channels = len(bands)
        self.register_buffer(
            'mean', torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1))
        self.register_buffer(
            'std', torch.tensor(IMAGENET_STD).view(1, 3, 1, 1))
        self.head = nn.Sequential(
            nn.Dropout(.4),
            nn.Linear(3 * FEATURE_DIM, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(.3),
            nn.Linear(256, 1),
        )

    def encode(self, image):
        image = F.interpolate(
            image, size=(self.resize_to, self.resize_to),
            mode='bilinear', align_corners=False)
        image = (image - self.mean) / self.std
        return self.backbone(image)

    def forward(self, images):
        pre = images[:, :self.half_channels]
        post = images[:, self.half_channels:]
        features = self.encode(torch.cat([pre, post], dim=0))
        pre_features, post_features = features.chunk(2, dim=0)
        joined = torch.cat([
            pre_features, post_features,
            torch.abs(post_features - pre_features),
        ], dim=1)
        return self.head(joined).squeeze(1)


def set_backbone_trainable(model, mode):
    for name, parameter in model.backbone.named_parameters():
        if mode == 'frozen':
            parameter.requires_grad = False
        elif mode == 'top':
            parameter.requires_grad = name.startswith(('layer3', 'layer4'))
        elif mode == 'all':
            parameter.requires_grad = True
        else:
            raise ValueError(f'Unknown backbone mode: {mode}')
    for module in model.backbone.modules():
        if isinstance(module, nn.BatchNorm2d):
            for parameter in module.parameters():
                parameter.requires_grad = False


def freeze_backbone_batchnorm(model):
    for module in model.backbone.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()


def cpu_state_dict(model):
    return {
        name: value.detach().cpu().clone()
        for name, value in model.state_dict().items()
    }


@torch.inference_mode()
def predict(model, loader):
    model.eval()
    probabilities, targets = [], []
    for images, yb in loader:
        with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(images.to(DEVICE, non_blocking=True))
        probabilities.append(torch.sigmoid(logits.float()).cpu().numpy())
        targets.append(yb.numpy())
    return np.concatenate(probabilities), np.concatenate(targets).astype(np.int8)


model = RspSiameseDamage(
    new_rsp_backbone(), CONFIG['resize_to'], CONFIG['bands']).to(DEVICE)


In [ ]:
def fit_phase(model, phase_name, phase_cfg, backbone_mode):
    """Fit, resume, or reuse one RSP training phase."""
    set_backbone_trainable(model, backbone_mode)
    parameters = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(
        parameters, lr=phase_cfg['learning_rate'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=.5,
        patience=phase_cfg['reduce_lr_patience'], min_lr=1e-7)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    signature = {
        'phase': phase_name,
        'phase_config': phase_cfg,
        'backbone_mode': backbone_mode,
        'rsp_sha256': rsp_sha256,
        'dataset_fingerprint': data['metadata']['preprocessing_fingerprint'],
        'bands': CONFIG['bands'],
        'normalization': normalization,
        'augmentation': CONFIG['augmentation'],
        'split': split_meta,
        'seed': CONFIG['seed'],
        'selection_metric': CONFIG['selection_metric'],
    }
    final_path = OUT['models'] / f'{phase_name}_best.pt'
    resume_path = OUT['models'] / f'{phase_name}_training.pt'

    if final_path.exists():
        state = torch.load(final_path, map_location='cpu', weights_only=False)
        if state.get('complete') and state.get('signature') == signature:
            model.load_state_dict(state['model_state'])
            print(
                f'reused {phase_name}: best epoch={state["best_epoch"]}, '
                f'val_ROC_AUC={state["best_val_ROC_AUC"]:.4f}')
            return {
                'phase': phase_name,
                'checkpoint': str(final_path),
                'validation_ROC_AUC': float(state['best_val_ROC_AUC']),
                'best_epoch': int(state['best_epoch']),
                'history': state['history'],
            }

    if SEALED_EXPERIMENT:
        raise RuntimeError(
            f'The experiment is sealed but {phase_name} has no compatible '
            'completed checkpoint. Use a new experiment_name.')

    start_epoch = 1
    best_auc, best_epoch, best_state = -np.inf, -1, None
    stale, history = 0, []
    if resume_path.exists():
        state = torch.load(resume_path, map_location='cpu', weights_only=False)
        if state.get('signature') == signature:
            model.load_state_dict(state['model_state'])
            optimizer.load_state_dict(state['optimizer_state'])
            scheduler.load_state_dict(state['scheduler_state'])
            if state.get('scaler_state'):
                scaler.load_state_dict(state['scaler_state'])
            start_epoch = int(state['completed_epoch']) + 1
            best_auc = float(state['best_val_ROC_AUC'])
            best_epoch = int(state['best_epoch'])
            best_state = state['best_model_state']
            stale = int(state['stale_epochs'])
            history = list(state['history'])
            torch.set_rng_state(state['torch_rng_state'])
            if USE_AMP and state.get('cuda_rng_state_all') is not None:
                torch.cuda.set_rng_state_all(state['cuda_rng_state_all'])
            if state.get('loader_rng_state') is not None:
                train_loader.generator.set_state(state['loader_rng_state'])
            print(
                f'resuming {phase_name} at epoch {start_epoch}; '
                f'best epoch={best_epoch}, val_ROC_AUC={best_auc:.4f}')

    epoch_range = (
        range(start_epoch, phase_cfg['epochs'] + 1)
        if stale < phase_cfg['patience'] else ())
    for epoch in epoch_range:
        started = time.time()
        model.train()
        freeze_backbone_batchnorm(model)
        loss_sum, seen = 0.0, 0
        for images, yb in train_loader:
            images = images.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits = model(images)
                loss = criterion(logits.float(), yb)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_sum += float(loss.detach()) * len(yb)
            seen += len(yb)

        val_scores, val_y = predict(model, val_loader)
        val_auc = float(roc_auc_score(val_y, val_scores))
        scheduler.step(val_auc)
        history.append({
            'phase': phase_name, 'epoch': epoch,
            'train_loss': loss_sum / max(seen, 1),
            'val_ROC_AUC': val_auc,
            'learning_rate': optimizer.param_groups[0]['lr'],
        })
        improved = best_state is None or val_auc > best_auc + 1e-6
        if improved:
            best_auc, best_epoch, stale = val_auc, epoch, 0
            best_state = cpu_state_dict(model)
        else:
            stale += 1

        atomic_torch_save({
            'complete': False,
            'signature': signature,
            'completed_epoch': epoch,
            'model_state': cpu_state_dict(model),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'scaler_state': scaler.state_dict(),
            'best_model_state': best_state,
            'best_val_ROC_AUC': float(best_auc),
            'best_epoch': int(best_epoch),
            'stale_epochs': int(stale),
            'history': history,
            'torch_rng_state': torch.get_rng_state(),
            'cuda_rng_state_all': (
                torch.cuda.get_rng_state_all() if USE_AMP else None),
            'loader_rng_state': train_loader.generator.get_state(),
        }, resume_path)
        print(
            f'{phase_name} epoch {epoch:03d} '
            f'loss={history[-1]["train_loss"]:.4f} '
            f'val_ROC_AUC={val_auc:.4f} best={best_auc:.4f} '
            f'lr={optimizer.param_groups[0]["lr"]:.2e} '
            f'time={time.time() - started:.0f}s'
            + (' *' if improved else ''))
        if stale >= phase_cfg['patience']:
            print(f'early stop; best epoch was {best_epoch}')
            break

    if best_state is None:
        raise RuntimeError(f'{phase_name} has no best model state')
    model.load_state_dict(best_state)
    atomic_torch_save({
        'complete': True,
        'signature': signature,
        'model_state': best_state,
        'best_val_ROC_AUC': float(best_auc),
        'best_epoch': int(best_epoch),
        'history': history,
    }, final_path)
    print(
        f'saved {phase_name}: best epoch={best_epoch}, '
        f'val_ROC_AUC={best_auc:.4f}')
    return {
        'phase': phase_name,
        'checkpoint': str(final_path),
        'validation_ROC_AUC': float(best_auc),
        'best_epoch': int(best_epoch),
        'history': history,
    }


In [ ]:
phase_results = {}
phase_results['phase1'] = fit_phase(
    model, 'phase1', CONFIG['phase1'], backbone_mode='frozen')

# Phase 2 always starts from the best frozen-backbone checkpoint unless an
# interrupted compatible phase-2 checkpoint resumes it further ahead.
phase1_state = torch.load(
    phase_results['phase1']['checkpoint'], map_location='cpu',
    weights_only=False)
model.load_state_dict(phase1_state['model_state'])
phase_results['phase2'] = fit_phase(
    model, 'phase2', CONFIG['phase2'],
    backbone_mode=CONFIG['phase2']['trainable_backbone'])

selected_phase = max(
    phase_results,
    key=lambda name: phase_results[name]['validation_ROC_AUC'])
selected_state = torch.load(
    phase_results[selected_phase]['checkpoint'], map_location='cpu',
    weights_only=False)
model.load_state_dict(selected_state['model_state'])
model.to(DEVICE).eval()

histories = []
for result in phase_results.values():
    histories.extend(result['history'])
history = pd.DataFrame(histories)
history.to_csv(OUT['metrics'] / 'training_history.csv', index=False)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for phase, frame in history.groupby('phase', sort=False):
    axes[0].plot(frame['epoch'], frame['train_loss'], label=phase)
    axes[1].plot(frame['epoch'], frame['val_ROC_AUC'], label=phase)
axes[0].set(title='training loss', xlabel='phase epoch')
axes[1].set(title='merged-validation ROC AUC', xlabel='phase epoch')
for ax in axes:
    ax.legend(); ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(OUT['figures'] / 'training_history.png', dpi=180, bbox_inches='tight')
plt.show()
print('selected phase:', selected_phase)


## 6. Validation calibration and diagnostics

The selected phase is fixed using ROC AUC. The threshold below is then calibrated separately by maximizing F1 on merged validation.


In [ ]:
val_probs, val_y = predict(model, val_loader)
threshold, validation_best_f1 = best_f1_threshold(val_y, val_probs)
validation_metrics = binary_metrics(val_y, val_probs, threshold)

validation_regions = []
for name, loader in (
    ('validation_merged', val_loader),
    ('validation_core', val_core_loader),
    ('validation_extra', val_extra_loader),
):
    scores, region_y = predict(model, loader)
    validation_regions.append({
        'region': name,
        **binary_metrics(region_y, scores, threshold),
    })
VALIDATION_REGIONS = pd.DataFrame(validation_regions)
VALIDATION_REGIONS.to_csv(
    OUT['metrics'] / 'validation_metrics_by_band.csv', index=False)
display(VALIDATION_REGIONS)

pd.DataFrame({
    'system_index': data['system_index'][idx_val],
    'y': val_y, 'score': val_probs,
    'prediction': (val_probs >= threshold).astype(np.int8),
}).to_parquet(
    OUT['predictions'] / 'validation_predictions.parquet', index=False)

precision, recall, _ = precision_recall_curve(val_y, val_probs)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(recall, precision,
        label=f'RSP ResNet-50 PR-AUC={validation_metrics["PR_AUC"]:.3f}')
ax.axhline(val_y.mean(), color='.5', linestyle='--', label='prevalence')
ax.set(xlabel='recall', ylabel='precision', title='Merged validation PR curve')
ax.legend(); ax.grid(alpha=.2)
fig.savefig(OUT['figures'] / 'validation_pr_curve.png', dpi=180, bbox_inches='tight')
plt.show()

computed_selection = {
    'model': 'RSP ResNet-50 Siamese RGB (MillionAID)',
    'phase': selected_phase,
    'validation_ROC_AUC': validation_metrics['ROC_AUC'],
    'validation_PR_AUC': validation_metrics['PR_AUC'],
    'selection_metric': 'validation_roc_auc',
    'threshold': threshold,
    'threshold_metric': CONFIG['threshold_metric'],
    'phase_checkpoint': phase_results[selected_phase]['checkpoint'],
    'rsp_weights': str(rsp_safe),
    'rsp_sha256': rsp_sha256,
    'normalization': normalization,
    'split': split_meta,
    'selected_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}
selected_model_path = OUT['models'] / 'selected_pretrained_model.pt'
computed_selection['saved_model'] = str(selected_model_path)
if FINAL_SELECTION_PATH.exists():
    with FINAL_SELECTION_PATH.open(encoding='utf-8') as handle:
        selection = json.load(handle)
    stable_fields = (
        'model', 'phase', 'selection_metric', 'threshold_metric', 'phase_checkpoint',
        'rsp_sha256', 'normalization', 'split', 'saved_model')
    same = all(
        selection.get(key) == computed_selection.get(key)
        for key in stable_fields)
    same = same and np.isclose(
        selection.get('threshold', np.nan),
        computed_selection['threshold'], rtol=0, atol=1e-10)
    if not same:
        raise RuntimeError(
            'The frozen selection differs from the current validation '
            'winner. Use a new experiment_name instead of replacing it.')
    if not selected_model_path.exists():
        raise FileNotFoundError(
            f'Frozen selected model is missing: {selected_model_path}')
    print('Reused the frozen validation selection.')
else:
    selection = computed_selection
    atomic_torch_save({
        'complete': True,
        'model_state': cpu_state_dict(model),
        'config': CONFIG,
        'selection': selection,
        'normalization': normalization,
        'split': split_meta,
    }, selected_model_path)
    atomic_json_dump(selection, FINAL_SELECTION_PATH)
    print('Saved the frozen validation selection.')
print(json.dumps(selection, indent=2))


## 7. One-shot evaluation on the identical sealed Planet test band

The test loader is first created here, after phase selection and threshold calibration are frozen. No test-optimal threshold is calculated.


In [ ]:
final_metrics_path = OUT['metrics'] / 'final_test_metrics.json'
final_predictions_path = (
    OUT['predictions'] / 'final_test_predictions.parquet')
if FINAL_MARKER_PATH.exists():
    with FINAL_MARKER_PATH.open(encoding='utf-8') as handle:
        marker = json.load(handle)
    if marker.get('selection') != selection:
        raise RuntimeError(
            'The final-test marker belongs to a different selection.')
    if not final_metrics_path.exists() or not final_predictions_path.exists():
        raise RuntimeError(
            'The final-test marker exists but a saved output is missing.')
    with final_metrics_path.open(encoding='utf-8') as handle:
        test_metrics = json.load(handle)
    print('Final test already sealed; loaded the saved metrics.')
else:
    test_loader = make_loader(idx_test)
    test_probs, test_y = predict(model, test_loader)
    test_metrics = binary_metrics(test_y, test_probs, threshold)
    test_predictions = (test_probs >= threshold).astype(np.int8)
    pd.DataFrame({
        'system_index': data['system_index'][idx_test],
        'y': test_y, 'score': test_probs, 'prediction': test_predictions,
    }).to_parquet(final_predictions_path, index=False)
    atomic_json_dump(test_metrics, final_metrics_path)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    matrix = confusion_matrix(test_y, test_predictions, labels=[0, 1])
    ConfusionMatrixDisplay(
        matrix, display_labels=['intact', 'damaged']).plot(
        ax=axes[0], cmap='Blues', colorbar=False)
    axes[0].set_title('RSP ResNet-50 final test confusion matrix')

    precision, recall, _ = precision_recall_curve(test_y, test_probs)
    axes[1].plot(
        recall, precision, label=f'PR-AUC={test_metrics["PR_AUC"]:.3f}')
    axes[1].axhline(
        test_y.mean(), color='.5', linestyle=':', label='test prevalence')
    axes[1].scatter(
        test_metrics['recall'], test_metrics['precision'], color='tab:red',
        zorder=3, label=f'frozen threshold={threshold:.3f}')
    axes[1].set(
        xlabel='recall', ylabel='precision',
        title='RSP ResNet-50 final test PR curve')
    axes[1].legend(); axes[1].grid(alpha=.2)
    fig.tight_layout()
    fig.savefig(OUT['figures'] / 'final_test_diagnostics.png',
                dpi=180, bbox_inches='tight')
    plt.show()

    # Confirm that the selected artifact contains the fine-tuned backbone.
    reloaded = RspSiameseDamage(
        torchvision.models.resnet50(weights=None),
        CONFIG['resize_to'], CONFIG['bands'])
    reloaded.backbone.fc = nn.Identity()
    reloaded.load_state_dict(torch.load(
        selection['saved_model'], map_location='cpu',
        weights_only=False)['model_state'])
    reloaded.to(DEVICE).eval()
    reload_probs, _ = predict(reloaded, test_loader)
    reload_error = float(np.max(np.abs(reload_probs - test_probs)))
    print(
        f'reload max absolute probability difference: {reload_error:.2e}')

    marker = {
        'selection': selection,
        'metrics': test_metrics,
        'prediction_file': final_predictions_path.name,
        'reload_max_abs_error': reload_error,
        'completed_utc': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    }
    atomic_json_dump(marker, FINAL_MARKER_PATH)
    print(f'Sealed final-test outputs in {FINAL_MARKER_PATH}.')

display(pd.DataFrame([test_metrics]))


## Comparison contract

This experiment's `final_test_metrics.json` can be compared with the scratch and other pretrained experiments. Their test prediction files join on `system_index` for paired error analysis. PR AUC and ROC AUC are the main cross-model ranking metrics because they do not depend on each model's validation-calibrated threshold.
